# Module 15 — Mixture of Experts (MoE)

**Part IV: Scaling & Efficiency**  •  *~35 minutes*

> *"Not every token needs every parameter. What if we had specialized sub-networks and a traffic cop?"*

Here's a fact that would have sounded absurd in 2022 but is now boring: **more than 60% of the frontier open models released in 2025–2026 are Mixture-of-Experts.** DeepSeek-V3, Llama-4 (Scout + Maverick), KIMI K2.5, Qwen3-MoE, Mixtral's descendants — they're all MoE. Dense scaling hit a wall. MoE punched through it.

In this module you'll build an MoE from scratch, break it on purpose (watch a routing network collapse to a single expert in front of your eyes), fix it with a load-balancing loss, and then look at how the big labs actually build these things in 2026.

### What you'll walk away with

1. An intuition for why MoE is the natural answer to "I want huge capacity but small FLOPs."
2. The router math, the top-k trick, and the dispatch-combine pattern.
3. A working 8-expert top-2 MoE in ~60 lines of PyTorch.
4. A first-hand feel for **expert collapse** and how load balancing prevents it.
5. A mental map of how DeepSeek-V3, Llama-4 Maverick, and KIMI K2.5 differ.


## 1. The problem: dense models waste parameters

When you run a dense transformer, **every single weight in the FFN fires for every single token.** The FFN is already 2/3 of the parameters in a modern transformer. You pay full price for "the" and full price for "photosynthesis."

Think about that for a second. If you were a brain, you wouldn't light up your motor cortex every time you read a word. You'd activate the relevant modules.

The MoE insight is almost embarrassingly simple:

> Replace the single big FFN with $N$ smaller FFNs ("experts"). For each token, use a tiny network (the "router") to decide which $k \ll N$ experts to run. Add their outputs up. Done.

That's it. That's the whole trick. Everything else — load balancing, shared experts, auxiliary losses, expert parallelism — is plumbing to make it actually work.


## 2. The math, slowly

Let a single token's hidden state be $x \in \mathbb{R}^d$. We have $N$ experts $E_1, \dots, E_N$, each a small FFN (say a 2-layer MLP). We also have a **router** parameterized by $W_g \in \mathbb{R}^{N \times d}$.

**Step 1 — score the experts:**
$$
s = W_g\, x \in \mathbb{R}^N
$$

**Step 2 — softmax into probabilities:**
$$
p = \mathrm{softmax}(s)
$$

**Step 3 — keep only the top-$k$:**
$$
G(x) = \mathrm{TopK}(p,\, k)
$$
where $G(x)$ is zero everywhere except the top-$k$ slots, and those slots are renormalized so they sum to 1.

**Step 4 — combine expert outputs:**
$$
y = \sum_{i=1}^{N} G(x)_i \cdot E_i(x)
$$

Only the $k$ experts with nonzero $G(x)_i$ actually need to run. That's the whole point. With $N=8, k=2$ you have the *knowledge* of 8 experts but pay the *compute* of 2. Multiply that by a trillion parameters and you get KIMI K2.5: **1T total, 32B active** — roughly 3% sparsity.


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

# Make matplotlib a little less ugly
plt.rcParams.update({
    "figure.figsize": (8, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
print("torch", torch.__version__)

## 3. A router in 5 lines

Before the full MoE, let's isolate the router. Given a batch of tokens, it produces a sparse gate vector per token. No experts yet — just the dispatcher.

In [ ]:
class Router(nn.Module):
    def __init__(self, d_model, n_experts, top_k=2):
        super().__init__()
        self.w_g = nn.Linear(d_model, n_experts, bias=False)
        self.n_experts = n_experts
        self.top_k = top_k

    def forward(self, x):
        # x: (B, d_model) or (B, T, d_model) — we'll flatten for simplicity
        logits = self.w_g(x)                     # (..., n_experts)
        probs = F.softmax(logits, dim=-1)
        topk_vals, topk_idx = probs.topk(self.top_k, dim=-1)
        # Renormalize so the kept weights sum to 1
        topk_vals = topk_vals / (topk_vals.sum(dim=-1, keepdim=True) + 1e-9)
        return topk_vals, topk_idx, probs  # also return full probs for diagnostics

router = Router(d_model=16, n_experts=8, top_k=2)
x = torch.randn(5, 16)
vals, idx, probs = router(x)
for t in range(5):
    picks = [(int(i), float(v)) for i, v in zip(idx[t], vals[t])]
    print(f"token {t}: picks = {picks}")

Each token picks two experts and splits its "vote" between them. Nothing is learned yet, so the picks are random-ish — but the mechanism is in place.

## 4. The experts

Each expert is just a small MLP — a gated FFN is common in practice (SwiGLU, GeGLU, etc.) but we'll use a plain 2-layer MLP to keep the math clear.

In [ ]:
class Expert(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

# Sanity check: 8 experts, each a 16 -> 64 -> 16 MLP
experts = nn.ModuleList([Expert(16, 64) for _ in range(8)])
print(f"Each expert: {sum(p.numel() for p in experts[0].parameters())} params")
print(f"8 experts total: {sum(p.numel() for e in experts for p in e.parameters())} params")

## 5. Putting it together — the toy MoE

The tricky bit in practice is **dispatch**: you don't want to run every expert on every token (that defeats the whole purpose). You want to gather the tokens routed to expert $i$, run expert $i$ once on that mini-batch, and scatter the results back.

For a toy notebook, readability beats speed. We'll loop over experts. Every production implementation (Mixtral, DeepSeek, vLLM's fused MoE kernel) does the same thing vectorized across GPUs.

In [ ]:
class MoE(nn.Module):
    def __init__(self, d_model, d_ff, n_experts, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = Router(d_model, n_experts, top_k)
        self.experts = nn.ModuleList([Expert(d_model, d_ff) for _ in range(n_experts)])

    def forward(self, x):
        # x: (B, d_model)
        B, d = x.shape
        gate_vals, gate_idx, full_probs = self.router(x)   # (B, k), (B, k), (B, N)

        y = torch.zeros_like(x)
        # For diagnostics
        expert_load = torch.zeros(self.n_experts)

        for e in range(self.n_experts):
            # Find every (token, slot) pair that routed to expert e
            mask = (gate_idx == e)                         # (B, k)
            if not mask.any():
                continue
            token_idx, slot_idx = mask.nonzero(as_tuple=True)
            weights = gate_vals[token_idx, slot_idx].unsqueeze(-1)  # (n, 1)
            inputs = x[token_idx]                           # (n, d)
            out = self.experts[e](inputs)
            y.index_add_(0, token_idx, weights * out)
            expert_load[e] = float(len(token_idx))

        return y, full_probs, expert_load

moe = MoE(d_model=16, d_ff=64, n_experts=8, top_k=2)
x = torch.randn(32, 16)
y, probs, load = moe(x)
print("input :", x.shape)
print("output:", y.shape)
print("load per expert:", load.tolist())
print(f"total dispatches: {int(load.sum())} (should be B*k = {32*2})")

Notice the load is lumpy. With a random-init router, a couple of experts are already getting more traffic than others. This is the seed of a disaster called **expert collapse** that we're about to watch unfold live.

## 6. A synthetic task to train on

We need something the router can plausibly specialize over. Let's build a toy task where the input is tagged with a *latent class* (0–3), and different classes need different functions.

- Class 0: $y = \sin(x)$
- Class 1: $y = x^2$
- Class 2: $y = \tanh(2x)$
- Class 3: $y = |x|$

The class is encoded into the first few dimensions of $x$, but it's mixed with noise so the router has to learn to find it. A good MoE should route each class to a different subset of experts.

In [ ]:
def make_batch(B, d_model=16):
    cls = torch.randint(0, 4, (B,))
    x = torch.randn(B, d_model) * 0.3
    # Embed the class as a one-hot in the first 4 dims
    x[torch.arange(B), cls] += 2.0
    # Target: a class-dependent nonlinear function of dim 4
    base = x[:, 4]
    y = torch.zeros(B, d_model)
    y[cls == 0, 4] = torch.sin(base[cls == 0])
    y[cls == 1, 4] = base[cls == 1] ** 2
    y[cls == 2, 4] = torch.tanh(2 * base[cls == 2])
    y[cls == 3, 4] = base[cls == 3].abs()
    return x, y, cls

x, y, cls = make_batch(8)
print("class labels:", cls.tolist())
print("x[0, :6]  :", x[0, :6].tolist())
print("y[0, :6]  :", y[0, :6].tolist())

## 7. Training run #1: no load balancing → collapse

Let's train the MoE with **only** the reconstruction loss $\|y - \hat y\|^2$ — no load balancing. Every step we'll log how much traffic each expert is getting.

In [ ]:
def train_moe(moe, steps=800, lb_alpha=0.0, lr=3e-3, log_every=20, verbose=True):
    opt = torch.optim.Adam(moe.parameters(), lr=lr)
    history = {"loss": [], "load": [], "lb_loss": []}
    for step in range(steps):
        x, y, _ = make_batch(128)
        y_hat, probs, load = moe(x)

        recon = F.mse_loss(y_hat, y)

        # Load balancing loss (see section 9 for the math).
        # We compute it every step but only USE it if lb_alpha > 0.
        f_i = load / (load.sum() + 1e-9)                # fraction of tokens to expert i
        p_i = probs.mean(dim=0)                          # mean routing prob to expert i
        lb = moe.n_experts * (f_i * p_i).sum()

        loss = recon + lb_alpha * lb

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % log_every == 0:
            history["loss"].append(float(recon))
            history["load"].append(load.clone())
            history["lb_loss"].append(float(lb))
            if verbose and step % (log_every * 10) == 0:
                print(f"step {step:4d} | recon {float(recon):.4f} | "
                      f"lb {float(lb):.3f} | load {load.int().tolist()}")
    return history

torch.manual_seed(1)
moe_bad = MoE(d_model=16, d_ff=64, n_experts=8, top_k=2)
hist_bad = train_moe(moe_bad, steps=800, lb_alpha=0.0)

Look at the load column. Within a couple hundred steps, most experts are getting **zero tokens**. The router discovered two or three experts that were marginally better at step 0 and kept slamming tokens into them. Those experts got even better. Positive feedback loop. Game over for the others.

This is **expert collapse**, and it's the classic MoE failure mode.

### Visualizing the collapse

Let's plot the load distribution over training. Each row = one logging step, each column = one expert, brightness = number of tokens routed there.

In [ ]:
def plot_load_heatmap(history, title):
    load = torch.stack(history["load"]).numpy()   # (T, N)
    fig, ax = plt.subplots(figsize=(8, 4))
    im = ax.imshow(load.T, aspect="auto", cmap="magma", interpolation="nearest")
    ax.set_xlabel("training step (×20)")
    ax.set_ylabel("expert id")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="tokens routed")
    plt.tight_layout()
    plt.show()

plot_load_heatmap(hist_bad, "No load balancing — expert collapse")

The bright horizontal streaks are the "winning" experts. The dark rows are the dead ones. Notice how quickly the darkness takes over — most experts are dead before step 200. Those parameters are wasted. You trained an 8-expert MoE and got, effectively, a 2-expert MoE with 6 expert-shaped bricks taped to the side.

## 8. Why does collapse happen?

It's a winner-take-all dynamic driven by two compounding effects:

1. **Gradient starvation.** An expert that never receives tokens gets **no gradient signal**. Its weights stay at initialization. It can never catch up.
2. **Router reinforcement.** Softmax routing is a competition. If expert 3 is 0.01 better right now, it gets more tokens, it learns faster, next step it's 0.02 better, it gets even more tokens...

This is exactly the instability the original Shazeer et al. (2017) "Outrageously Large Neural Networks" paper flagged, and every MoE architecture since has needed some mechanism to kill it.


## 9. The fix: load-balancing loss

The standard trick (Switch Transformer, GShard, Mixtral) is an **auxiliary loss** that pushes the router toward uniform usage. Let

- $f_i$ = fraction of tokens *actually* routed to expert $i$ (a discrete count),
- $p_i$ = average routing *probability* for expert $i$ (a soft, differentiable quantity).

Then:
$$
\mathcal{L}_{\text{bal}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot p_i
$$

Why does this work?

- If the router is perfectly balanced, $f_i = p_i = 1/N$ and the loss equals $\alpha$. (A fixed constant — no gradient pressure.)
- If expert $i$ is overused, $f_i$ and $p_i$ are both large, so their product is *much* larger, and the gradient pushes $p_i$ down.
- $f_i$ is the "stop-gradient" counting term; $p_i$ is where the gradient actually flows.

Typical $\alpha \approx 0.01$. Small enough that it doesn't fight the task loss, big enough to keep everyone employed.


## 10. Training run #2: with load balancing

In [ ]:
torch.manual_seed(1)
moe_good = MoE(d_model=16, d_ff=64, n_experts=8, top_k=2)
hist_good = train_moe(moe_good, steps=800, lb_alpha=0.01)

In [ ]:
plot_load_heatmap(hist_good, "With load balancing α=0.01 — stays balanced")

Night and day. Every expert stays lit up. The load isn't perfectly uniform — and that's fine, we *want* some specialization — but no expert is starving.

### Side-by-side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, hist, title in [
    (axes[0], hist_bad,  "no balancing (collapsed)"),
    (axes[1], hist_good, "α = 0.01 (healthy)"),
]:
    load = torch.stack(hist["load"]).numpy()
    im = ax.imshow(load.T, aspect="auto", cmap="magma", interpolation="nearest")
    ax.set_title(title)
    ax.set_xlabel("step (×20)")
axes[0].set_ylabel("expert id")
plt.tight_layout()
plt.show()

In [ ]:
# Loss curves
fig, ax = plt.subplots()
ax.plot(hist_bad["loss"],  label="no balancing",   lw=2)
ax.plot(hist_good["loss"], label="α = 0.01",        lw=2)
ax.set_xlabel("step (×20)")
ax.set_ylabel("reconstruction loss")
ax.set_title("Task loss: balanced MoE learns faster")
ax.legend()
plt.tight_layout()
plt.show()

The balanced model not only stays healthier, it actually **learns the task faster**. That makes sense: it has 8 functioning experts instead of 2, so it has 4x the effective capacity.

## 11. Did the experts specialize?

We trained on 4 latent classes with 8 experts. Did the balanced router learn to send each class to distinct experts? Let's plot a confusion-style matrix: rows = classes, columns = experts, cell = fraction of tokens of that class routed through that expert.

In [ ]:
@torch.no_grad()
def routing_by_class(moe, batches=20, B=256):
    n = moe.n_experts
    counts = torch.zeros(4, n)
    for _ in range(batches):
        x, _, cls = make_batch(B)
        _, probs, _ = moe(x)                       # (B, N) full soft probs
        for c in range(4):
            sel = probs[cls == c]
            if len(sel):
                counts[c] += sel.sum(dim=0)
    # Normalize each row to a probability
    counts = counts / counts.sum(dim=1, keepdim=True)
    return counts

spec_good = routing_by_class(moe_good)
spec_bad  = routing_by_class(moe_bad)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, mat, title in [
    (axes[0], spec_bad,  "collapsed model"),
    (axes[1], spec_good, "balanced model"),
]:
    im = ax.imshow(mat.numpy(), cmap="viridis", vmin=0, vmax=mat.max())
    ax.set_xticks(range(8))
    ax.set_yticks(range(4))
    ax.set_yticklabels([f"sin", "sq", "tanh", "abs"])
    ax.set_xlabel("expert id")
    ax.set_title(title)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j].item()
            if v > 0.05:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v < 0.4 else "black", fontsize=8)
axes[0].set_ylabel("class")
plt.suptitle("Routing distribution by class — who handles what?", y=1.02)
plt.tight_layout()
plt.show()

In the balanced model you should see clear "stripes" — a small group of experts lights up for each class, with little overlap. That's specialization emerging *from the router alone*, with no explicit supervision telling it which class is which. The collapsed model, by contrast, routes everything through the same 1–2 experts regardless of class.

This is the property that makes MoE work in the wild: **experts implicitly discover topic/domain structure** (code, math, Chinese, SQL, etc.) during pretraining. Nobody tells them to.


## 12. The parameter accounting trick

Let's count what we just built and extrapolate:


In [ ]:
def count_params(moe):
    n_router = sum(p.numel() for p in moe.router.parameters())
    n_per_expert = sum(p.numel() for p in moe.experts[0].parameters())
    n_total = n_router + n_per_expert * moe.n_experts
    n_active = n_router + n_per_expert * moe.top_k
    return n_router, n_per_expert, n_total, n_active

r, pe, total, active = count_params(moe_good)
print(f"router          : {r:,}")
print(f"per expert      : {pe:,}")
print(f"total  (N=8)    : {total:,}")
print(f"active (k=2)    : {active:,}")
print(f"sparsity factor : {total / active:.2f}x  (knowledge / compute ratio)")

Now imagine scaling this the way the real models do. If each expert were a full transformer FFN block with $d_{ff} = 4 d_{model}$, and we had say $d_{model} = 7168$ (KIMI-scale), 384 experts, top-8 routing:

$$
\frac{\text{total}}{\text{active}} = \frac{384}{8} = 48
$$

That's the magic number. **KIMI K2.5 is a 1T-parameter model that costs 32B per token to run.** Same ratio.

In [ ]:
# A little calculator for the frontier models
frontier = [
    ("Mixtral 8x7B",        "8 × 7B",   47,  13),   # total B, active B
    ("DeepSeek-V3",         "256 + 1 shared", 671, 37),
    ("Llama-4 Maverick",    "128 routed + 1 shared (alt dense)", 400, 17),
    ("KIMI K2.5",           "384 experts, top-8", 1000, 32),
]
print(f"{'model':<22}{'layout':<36}{'total B':>10}{'active B':>12}{'sparsity':>12}")
print("-" * 92)
for name, layout, tot, act in frontier:
    print(f"{name:<22}{layout:<36}{tot:>10}{act:>12}{tot/act:>11.1f}x")

## 13. How the frontier labs actually do it (2026)

The toy MoE you just built is roughly 2017-Shazeer. Here's what changed since.

### 13a. DeepSeek-V3 — fine-grained + shared experts

DeepSeek-V3 has **671B total / 37B active**, and two innovations worth knowing:

1. **Shared experts.** A small number of "shared" experts are run on *every* token, unconditionally. Only the *routed* experts (256 of them) go through top-k. The shared experts learn the "boring general stuff" that every token needs (grammar, basic syntax), freeing the routed experts to specialize harder.
2. **Fine-grained experts.** Instead of 8 fat experts, use 256 skinny ones. This dramatically increases the number of *subsets* of experts the router can select — more combinatorial expressivity from the same parameter budget.
3. **Auxiliary-loss-free balancing.** V3 dropped the $\mathcal{L}_{\text{bal}}$ loss entirely. Instead it maintains a bias term on each router logit that gets nudged up or down as experts become over/under-loaded. No gradient interference with the task loss. Cleaner.

### 13b. Llama-4 Maverick — alternating MoE/dense

Maverick (~400B total / 17B active, 128 routed + 1 shared expert) does something different: **half the layers are dense, half are MoE**, in strict alternation. The thinking: dense layers preserve general representational coherence, MoE layers provide specialized knowledge. Best of both.

Surprisingly, Maverick uses **top-1** routing (each token picks *one* routed expert plus the shared one), with a sparsity factor of 64.

### 13c. KIMI K2.5 — the extreme end

KIMI K2.5 (Moonshot, 2026) pushes the ratio: **1T total, 32B active**, with **384 experts and top-8 routing**. That's 3.2% of parameters active per token. Moonshot's report argues that past a certain total size, the sparsity factor is the dominant lever — at 1T dense, nobody could afford to serve the model.

### 13d. CMoE / "carved" MoE

A 2025 line of work shows you can **carve** an MoE out of an already-trained *dense* model without pretraining from scratch. You cluster the FFN neurons into groups, treat each group as an expert, and fine-tune a router. It's a cheap way to turn a dense model you already own into a sparse one.

The through-line across all of these: **routing + load balancing + expert specialization** — the same three ideas you just implemented, dressed up in different outfits.


## 14. "Break things on purpose" — three quick experiments

The best way to build intuition is to poke at the working system and watch it degrade. Run each of these on `moe_good` and see what happens.

In [ ]:
# Experiment A: what if we force top-k = 1?
class Top1MoE(MoE):
    def __init__(self, *args, **kwargs):
        kwargs["top_k"] = 1
        super().__init__(*args, **kwargs)

torch.manual_seed(1)
moe_k1 = MoE(16, 64, n_experts=8, top_k=1)
hist_k1 = train_moe(moe_k1, steps=800, lb_alpha=0.01, verbose=False)

torch.manual_seed(1)
moe_k4 = MoE(16, 64, n_experts=8, top_k=4)
hist_k4 = train_moe(moe_k4, steps=800, lb_alpha=0.01, verbose=False)

fig, ax = plt.subplots()
ax.plot(hist_k1["loss"],   label="top-1")
ax.plot(hist_good["loss"], label="top-2")
ax.plot(hist_k4["loss"],   label="top-4")
ax.set_xlabel("step (×20)")
ax.set_ylabel("reconstruction loss")
ax.set_title("Effect of k on learning")
ax.legend()
plt.tight_layout()
plt.show()

Top-1 is hard to train (each token has a single expert, no soft blending — harsh discrete decisions). Top-4 is softer and usually converges better but costs 2x the FLOPs of top-2. **Top-2 is the sweet spot** for most models, which is why it's the default almost everywhere.

In [ ]:
# Experiment B: what if the router sees only part of the input?
# (Simulating "cheap router" — literally 0 new parameters for routing)
class NoRouter(nn.Module):
    # Router that routes purely by random projection. Not learnable.
    def __init__(self, n_experts, top_k):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
    def forward(self, x):
        # Hash-ish: project onto fixed random directions
        if not hasattr(self, "proj"):
            self.proj = torch.randn(x.shape[-1], self.n_experts) / math.sqrt(x.shape[-1])
        logits = x @ self.proj
        probs = F.softmax(logits, dim=-1)
        v, i = probs.topk(self.top_k, dim=-1)
        v = v / (v.sum(dim=-1, keepdim=True) + 1e-9)
        return v, i, probs

# A hash-routed MoE: no learnable routing
torch.manual_seed(1)
moe_hash = MoE(16, 64, n_experts=8, top_k=2)
moe_hash.router = NoRouter(8, 2)
hist_hash = train_moe(moe_hash, steps=800, lb_alpha=0.0, verbose=False)

fig, ax = plt.subplots()
ax.plot(hist_good["loss"], label="learned router")
ax.plot(hist_hash["loss"], label="random-projection router")
ax.set_xlabel("step (×20)")
ax.set_ylabel("loss")
ax.set_title("Learning to route matters (a lot)")
ax.legend()
plt.tight_layout()
plt.show()

A **learned** router is what makes MoE work. Random-hash routing is a well-known cheap baseline (e.g. HashLayer from Facebook, 2021) and it does surprisingly OK at scale — but on small models like this, the learned router wins handily.

In [ ]:
# Experiment C: crank the load balancing α way up
torch.manual_seed(1)
moe_over = MoE(16, 64, n_experts=8, top_k=2)
hist_over = train_moe(moe_over, steps=800, lb_alpha=1.0, verbose=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist_good["loss"], label="α = 0.01 (good)")
axes[0].plot(hist_over["loss"], label="α = 1.0  (too much)")
axes[0].set_title("Task loss")
axes[0].set_xlabel("step (×20)"); axes[0].legend()

load = torch.stack(hist_over["load"]).numpy()
axes[1].imshow(load.T, aspect="auto", cmap="magma")
axes[1].set_title("Load heatmap at α = 1.0 (uniform but unspecialized)")
axes[1].set_xlabel("step (×20)"); axes[1].set_ylabel("expert id")
plt.tight_layout()
plt.show()

At $\alpha = 1.0$ the load is perfectly uniform, but the model is *worse at the task*. That's the trade-off in a single picture: **too little balancing → collapse; too much → no specialization.** You want just enough to keep everyone employed and then let the task loss do its thing.

## 15. Checkpoint quiz

Try these without scrolling back up.

1. An MoE has $N=64$ experts and uses top-4 routing. What's the ratio of total to active parameters, ignoring the router itself?

2. You train an MoE and after 1000 steps you find that 10 of 32 experts have load 0. What's the most likely cause and what's the standard fix?

3. Why does the load-balancing loss use $N \cdot \sum_i f_i \, p_i$ instead of $\sum_i (f_i - 1/N)^2$?

4. DeepSeek-V3 has 256 routed experts but activates 8 per token, *plus* 1 shared expert. The shared expert runs every time. What's it for?

5. KIMI K2.5 is "1T total / 32B active." If you serve it on a single H200 node, which number matters for your GPU RAM budget, and which for your latency budget?

---

**Answers:**

1. $64/4 = 16$x. The model carries 16x more "knowledge" than it computes per token.

2. **Expert collapse** — those experts got no gradient signal early in training and never recovered. Add a load-balancing auxiliary loss (classic) or DeepSeek-V3's bias-update trick (modern alternative).

3. $(f_i - 1/N)^2$ would penalize deviations symmetrically, but $f_i$ is a *count* (non-differentiable) — you can't backprop through it. The $f_i \, p_i$ form uses $f_i$ as a stop-gradient weight and lets the gradient flow through $p_i$ only, pushing probability mass *away from* whatever is currently overloaded. Much cleaner in practice.

4. The shared expert handles the **low-level, everybody-needs-it work** (basic grammar, tokenization-level smoothing). By separating "always on" from "routed," the routed experts are free to specialize hard on the interesting stuff without worrying about the basics.

5. **RAM** is governed by **total** (1T × bytes-per-param — you need the whole thing in memory to serve it). **Latency** is governed by **active** (32B worth of matmuls per forward pass, plus the routing overhead). This is exactly why MoE is so attractive at the frontier: you pay once for the RAM, but every query gets the small-model speed.


## 16. Where to go next

- **Module 16 — Quantization (GPTQ → AWQ → TurboQuant → RotorQuant).** Once you have a trillion-parameter MoE, you need every bit-packing trick in the book.
- **Module 29 — Distributed Inference.** MoE gets extra fun when experts live on different GPUs. That's called *expert parallelism* and it's the reason KIMI K2.5 can be served at all.
- **Go read the DeepSeek-V3 tech report.** It's the clearest single document on modern MoE, and the auxiliary-loss-free balancing trick is genuinely clever.
- **Try training your own MoE on real text.** Swap `make_batch` for a subset of TinyStories, scale the experts to real FFN width, and watch emergent specialization on linguistic features.

### The takeaway

MoE is not exotic anymore. It's the default. Once you internalize *"router + experts + load balancing"* as a single concept, every frontier architecture paper from 2024 onward becomes 80% legible. The remaining 20% is plumbing — fine-grained experts, shared experts, alternating layers, bias-based balancing — variations on the same three-part theme you just built from scratch.

Sparsity is how you scale past the wall.
